# Teach LightMoon's own model

This teaches a small AI (Qwen3 1.7B) to answer the way LightMoon's big AI does, from the training file LightMoon
made (Settings → AI brain → *Download training data*). At the end you get one file,
`lightmoon-YYYY-MM-DD.Q4_K_M.gguf` (about 1.1 GB), that runs on your computer, and later on your phone.

**On Kaggle**
1. On the right, under *Session options*: Accelerator **GPU T4 x2**, and Internet **On** (Kaggle asks you to
   verify your phone number once before it allows internet).
2. *+ Add Input* → *Upload* → choose your `lightmoon-training-….jsonl` file (or several: one from each device).
   Give the dataset any name.
3. Press **Run All**. It takes from about half an hour to a few hours, depending on how much you've collected.
4. When the last cell says **Done**, open *Output* on the right and download the `.gguf` file.

**On Google Colab** (if Kaggle isn't available): *Runtime* → *Change runtime type* → **T4 GPU**, then
*Runtime* → *Run all*. It asks for the training file, and downloads the `.gguf` file at the end.

Nothing here changes LightMoon itself: the file only becomes its model once you put it in place on your computer
(see `trainer/README.md`).

In [ ]:
# Settings. The defaults are fine; change them only if you know why.
BASE_MODEL = "unsloth/Qwen3-1.7B"  # or "unsloth/Qwen2.5-1.5B-Instruct": a little smaller, not as good with tools
MAX_LEN = 8192        # the longest conversation it learns from, in tokens (LightMoon's instructions are long)
MAX_TRAIN = 3000      # at most this many answers to learn from (the most recent)
EVAL_CASES = 120      # answers held back from training, to score it before and after
LEARNING_RATE = 2e-4

In [ ]:
# The training tools (a few minutes).
!pip install -q unsloth 2>&1 | tail -n 3

In [ ]:
%%writefile eval.py
"""How often our model does what the big model did, on examples it never trained on.

The rule is the same as LightMoon's own comparison (docs/js/learning.js):
- The big model called tools: our model must call the same tools, and the arguments that matter must agree
  (the required ones and any time; ids, numbers and choices exactly, times to the minute, text with mostly the
  same words).
- The big model answered in words: our model must answer in words too, sharing at least 40% of its words.
Each case counts under every kind either model chose (a tool's name, "talk", "after-tool", "briefing"), so a
kind's score also drops when our model does that kind when it shouldn't.

The training notebook uses this file to hold back one example in ten and to score the model before and after
training. On a computer, it can score our model while llama-server runs it (standard library only):

    python eval.py lightmoon-training-2026-11-02.jsonl --server http://127.0.0.1:8080/v1
"""
import argparse
import hashlib
import json
import re
import sys
import urllib.request
from datetime import datetime

SIMILAR_REPLY = 0.4
SIMILAR_TEXT = 0.5
MAX_STEPS = 4
HELD_OUT_EVERY = 10  # one example in ten is never trained on
BRIEFING_REQUEST = re.compile(r"^\(Automatic message: I just opened the app")
TIME_KEY = re.compile(r"(_at|_date|^start|^date)$")
KIND_NAMES = {"talk": "Just talking", "after-tool": "Confirming what it did", "briefing": "Spoken briefings", "writing": "Other writing"}


# ---------- The examples ----------

def load_examples(paths):
    """Every example in the files, once (the same conversation exported twice counts once)."""
    seen, examples = set(), []
    for path in paths:
        with open(path, encoding="utf-8") as file:
            for line in file:
                line = line.strip()
                if not line:
                    continue
                example = json.loads(line)
                key = example_id(example)
                if key not in seen:
                    seen.add(key)
                    examples.append(example)
    return examples


def example_id(example):
    return hashlib.sha256(json.dumps(example["messages"], sort_keys=True, ensure_ascii=False).encode("utf-8")).hexdigest()


def is_held_out(example):
    return int(example_id(example)[:8], 16) % HELD_OUT_EVERY == 0


def cases(example):
    """Each answer the big model gave after the last thing the user said, with everything before it."""
    messages, tools = example["messages"], example.get("tools")
    last_user = max((i for i, m in enumerate(messages) if m["role"] == "user"), default=-1)
    steps = [i for i, m in enumerate(messages) if i > last_user and m["role"] == "assistant"][:MAX_STEPS]
    return [(messages[:i], messages[i], tools) for i in steps]


# ---------- Comparing two answers ----------

def words(text):
    return set(re.findall(r"(?:[^\W_]|')+", str(text or "").lower()))


def similarity(a, b):
    """Shared words over all words (Dice), from 0 to 1."""
    x, y = words(a), words(b)
    if not x and not y:
        return 1.0
    return 2 * len(x & y) / (len(x) + len(y))


def tool_names(message):
    return sorted((call.get("function") or {}).get("name") or "" for call in message.get("tool_calls") or [])


def kinds_of(message, tools, prefix):
    names = tool_names(message)
    if names:
        return sorted({f"tool:{name}" for name in names})
    if not tools:
        last_user = next((m.get("content") or "" for m in reversed(prefix) if m["role"] == "user"), "")
        return ["briefing" if BRIEFING_REQUEST.match(last_user) else "writing"]
    return ["after-tool" if prefix and prefix[-1]["role"] == "tool" else "talk"]


def parse_args(value):
    if isinstance(value, dict):
        return value
    try:
        parsed = json.loads(value or "{}")
        return parsed if isinstance(parsed, dict) else {}
    except (TypeError, ValueError):
        return {}


def normalize_time(value):
    text = str(value).strip()
    if re.fullmatch(r"\d{4}-\d{2}-\d{2}", text):
        return text
    try:
        moment = datetime.fromisoformat(text.replace(" ", "T").replace("Z", "+00:00"))
    except ValueError:
        return None
    if moment.tzinfo:
        moment = moment.astimezone().replace(tzinfo=None)
    return moment.strftime("%Y-%m-%dT%H:%M")


def bare_id(value):
    return re.sub(r"\]$", "", re.sub(r"^\[?#", "", str(value)))


def arguments_agree(name, expected_args, answer_args, tools):
    schema = next(((t.get("function") or {}).get("parameters") or {} for t in tools or [] if (t.get("function") or {}).get("name") == name), {})
    properties = schema.get("properties") or {}
    keys = set(schema.get("required") or []) | {key for key in properties if TIME_KEY.search(key)}
    expected, answer = parse_args(expected_args), parse_args(answer_args)
    for key in keys:
        a, b = expected.get(key), answer.get(key)
        if a is None or a == "":
            if not (b is None or b == ""):
                return False
            continue
        if b is None:
            return False
        spec = properties.get(key) or {}
        if TIME_KEY.search(key):
            same = normalize_time(a) == normalize_time(b)
        elif key == "id" or key.endswith("_id") or key == "reply_to":
            same = bare_id(a) == bare_id(b)
        elif spec.get("type") == "string" and not spec.get("enum"):
            same = similarity(a, b) >= SIMILAR_TEXT
        else:
            same = str(a).lower() == str(b).lower()
        if not same:
            return False
    return True


def compare(expected, answer, tools, prefix):
    """One case: {"kinds": [...], "agree": bool, "similar": float or None}."""
    kinds = sorted(set(kinds_of(expected, tools, prefix)) | set(kinds_of(answer, tools, prefix)))
    want, got = tool_names(expected), tool_names(answer)
    similar = None
    if want or got:
        agree = want == got
        for name in want if agree else []:
            a = next(c for c in expected["tool_calls"] if c["function"]["name"] == name)
            b = next(c for c in answer["tool_calls"] if c["function"]["name"] == name)
            agree = agree and arguments_agree(name, a["function"].get("arguments"), b["function"].get("arguments"), tools)
    else:
        similar = similarity(expected.get("content"), answer.get("content"))
        agree = similar >= SIMILAR_REPLY
    return {"kinds": kinds, "agree": agree, "similar": similar}


# ---------- Getting our model's answer ----------

def parse_generated(text):
    """A raw reply in Qwen's format (text, and tool calls as <tool_call>{"name", "arguments"}</tool_call>)."""
    text = re.sub(r"<think>.*?</think>", "", text, flags=re.S)
    text = text.split("<|im_end|>")[0]
    calls = []
    for i, raw in enumerate(re.findall(r"<tool_call>\s*(.*?)\s*</tool_call>", text, flags=re.S)):
        try:
            call = json.loads(raw)
        except ValueError:
            continue
        arguments = call.get("arguments", call.get("parameters", {}))
        calls.append({"id": f"gen_{i}", "type": "function", "function": {
            "name": call.get("name", ""), "arguments": arguments if isinstance(arguments, str) else json.dumps(arguments, ensure_ascii=False)}})
    content = re.sub(r"<tool_call>.*?</tool_call>", "", text, flags=re.S).strip()
    return {"role": "assistant", "content": content, **({"tool_calls": calls} if calls else {})}


def ask_server(server, prefix, tools, timeout=300):
    """Our model's answer from an OpenAI-compatible server such as llama-server."""
    body = {"model": "local", "messages": prefix, "temperature": 0.2, "max_tokens": 400, "stream": False,
            "chat_template_kwargs": {"enable_thinking": False}}
    if tools:
        body.update(tools=tools, tool_choice="auto")
    request = urllib.request.Request(f"{server.rstrip('/')}/chat/completions", data=json.dumps(body).encode("utf-8"),
                                     headers={"Content-Type": "application/json"})
    with urllib.request.urlopen(request, timeout=timeout) as response:
        message = json.load(response)["choices"][0]["message"]
    content = re.sub(r"<think>.*?</think>\s*", "", message.get("content") or "", flags=re.S).strip()
    calls = [{"id": c.get("id") or f"local_{i}", "type": "function", "function": {
        "name": c["function"]["name"],
        "arguments": c["function"]["arguments"] if isinstance(c["function"]["arguments"], str) else json.dumps(c["function"]["arguments"])}}
        for i, c in enumerate(message.get("tool_calls") or [])]
    return {"role": "assistant", "content": content, **({"tool_calls": calls} if calls else {})}


# ---------- Adding it up ----------

class Tally:
    def __init__(self):
        self.kinds = {}
        self.cases = 0
        self.agreed = 0

    def add(self, result):
        self.cases += 1
        self.agreed += result["agree"]
        for kind in result["kinds"]:
            tally = self.kinds.setdefault(kind, {"cases": 0, "agreed": 0, "similar": 0.0, "talks": 0})
            tally["cases"] += 1
            tally["agreed"] += result["agree"]
            if result["similar"] is not None:
                tally["similar"] += result["similar"]
                tally["talks"] += 1

    def overall(self):
        return self.agreed / self.cases if self.cases else 0.0

    def report(self, title="Our model"):
        lines = [f"{title}: agreed with the big model in {self.agreed} of {self.cases} cases ({100 * self.overall():.0f}%)"]
        for kind, tally in sorted(self.kinds.items(), key=lambda item: -item[1]["cases"]):
            name = KIND_NAMES.get(kind, kind.removeprefix("tool:").replace("_", " ").capitalize())
            alike = f", replies {100 * tally['similar'] / tally['talks']:.0f}% alike" if tally["talks"] else ""
            lines.append(f"  {name:<28} {tally['agreed']:>4} of {tally['cases']:<4} {100 * tally['agreed'] / tally['cases']:>4.0f}%{alike}")
        return "\n".join(lines)

    def as_dict(self):
        return {"cases": self.cases, "agreed": self.agreed, "kinds": self.kinds}


def main():
    parser = argparse.ArgumentParser(description="Score our model against the big model's answers.")
    parser.add_argument("files", nargs="+", help="training data from LightMoon (.jsonl)")
    parser.add_argument("--server", default="http://127.0.0.1:8080/v1", help="our model's address (llama-server)")
    parser.add_argument("--all", action="store_true", help="every example, not only the ones held back from training")
    parser.add_argument("--limit", type=int, default=200, help="at most this many cases")
    parser.add_argument("--report", default="eval-report.json")
    options = parser.parse_args()

    examples = [e for e in load_examples(options.files) if options.all or is_held_out(e)]
    todo = [case for example in examples for case in cases(example)][: options.limit]
    if not todo:
        sys.exit("No held-back examples to score yet (one in ten is held back). Try --all.")
    tally = Tally()
    for n, (prefix, expected, tools) in enumerate(todo, 1):
        try:
            answer = ask_server(options.server, prefix, tools)
        except OSError as err:
            sys.exit(f"Can't reach our model at {options.server}: {err}")
        tally.add(compare(expected, answer, tools, prefix))
        print(f"\r{n} of {len(todo)} cases", end="", file=sys.stderr, flush=True)
    print(file=sys.stderr)
    print(tally.report())
    with open(options.report, "w", encoding="utf-8") as file:
        json.dump(tally.as_dict(), file, indent=2)


if __name__ == "__main__":
    main()


In [ ]:
# Your training file(s): every .jsonl added as input (Kaggle), or uploaded now (Colab).
import glob, json, os, random, shutil, subprocess, sys, time
from datetime import date
import eval as ev

IN_COLAB = "google.colab" in sys.modules
WORK = "/kaggle/working" if os.path.isdir("/kaggle/working") else "/content"
files = sorted(glob.glob("/kaggle/input/**/*.jsonl", recursive=True) + glob.glob("/content/*.jsonl"))
if not files and IN_COLAB:
    from google.colab import files as colab_files
    print("Choose your LightMoon training file(s):")
    files = [os.path.abspath(name) for name in colab_files.upload()]
if not files:
    raise SystemExit("No training file found. Add it with '+ Add Input' -> 'Upload', then press Run All again.")

examples = ev.load_examples(files)
train_examples = [e for e in examples if not ev.is_held_out(e)]
held_out = [e for e in examples if ev.is_held_out(e)]
train_cases = [case for e in train_examples for case in ev.cases(e)][-MAX_TRAIN:]
eval_cases = [case for e in held_out for case in ev.cases(e)][:EVAL_CASES]
print(f"{len(examples)} conversations from {len(files)} file(s): {len(train_cases)} answers to learn from, "
      f"{len(eval_cases)} held back to test it.")
if len(train_cases) < 30:
    raise SystemExit("That's too few to learn from yet. Let LightMoon collect for a while longer (a few hundred is good).")
if len(eval_cases) < 20:
    print("Few answers are held back for testing, so the scores below will be rough.")

In [ ]:
# The model to teach.
from unsloth import FastLanguageModel
import torch

model, tokenizer = FastLanguageModel.from_pretrained(BASE_MODEL, max_seq_length=MAX_LEN, load_in_4bit=False, dtype=None)
model = FastLanguageModel.get_peft_model(
    model, r=16, lora_alpha=16, lora_dropout=0, bias="none", random_state=3407,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    use_gradient_checkpointing="unsloth",
)

In [ ]:
# Each answer the big model gave becomes one lesson: everything before it is context, the answer is what it learns.
ASSISTANT_MARK = "<|im_start|>assistant\n"

def for_template(messages):
    """Tool-call arguments as objects (what chat templates expect), and no missing text."""
    out = []
    for m in messages:
        m = {**m, "content": m.get("content") or ""}
        if m.get("tool_calls"):
            m["tool_calls"] = [{**c, "function": {**c["function"], "arguments": ev.parse_args(c["function"].get("arguments"))}}
                               for c in m["tool_calls"]]
        out.append(m)
    return out

def render(messages, tools, generation_prompt):
    # enable_thinking=False: answer straight away, as LightMoon asks llama-server to.
    return tokenizer.apply_chat_template(for_template(messages), tools=tools or None, tokenize=False,
                                         add_generation_prompt=generation_prompt, enable_thinking=False)

def lesson(prefix, expected, tools):
    prompt = render(prefix, tools, True)
    full = render(prefix + [expected], tools, False)
    split = len(prompt) if full.startswith(prompt) else full.rfind(ASSISTANT_MARK) + len(ASSISTANT_MARK)
    context = tokenizer(full[:split], add_special_tokens=False)["input_ids"]
    answer = tokenizer(full[split:], add_special_tokens=False)["input_ids"]
    if len(context) + len(answer) > MAX_LEN:
        return None
    return {"input_ids": context + answer, "attention_mask": [1] * (len(context) + len(answer)),
            "labels": [-100] * len(context) + answer}

lessons = [x for x in (lesson(*case) for case in train_cases) if x]
random.Random(3407).shuffle(lessons)
EPOCHS = 2 if len(lessons) < 1000 else 1
print(f"{len(lessons)} lessons ({len(train_cases) - len(lessons)} too long, skipped), {EPOCHS} pass(es) through them.")

In [ ]:
# How often it does what the big model did, on answers it never learned from (the same rule as LightMoon's).
def score(title):
    if not eval_cases:
        print("Nothing held back to test with yet.")
        return None
    FastLanguageModel.for_inference(model)
    tally = ev.Tally()
    for n, (prefix, expected, tools) in enumerate(eval_cases, 1):
        ids = tokenizer(render(prefix, tools, True), add_special_tokens=False, return_tensors="pt")["input_ids"]
        if ids.shape[1] > MAX_LEN - 256:
            continue
        with torch.no_grad():
            out = model.generate(input_ids=ids.to(model.device), attention_mask=torch.ones_like(ids).to(model.device),
                                 max_new_tokens=256, do_sample=False, pad_token_id=tokenizer.eos_token_id)
        text = tokenizer.decode(out[0, ids.shape[1]:], skip_special_tokens=False)
        tally.add(ev.compare(expected, ev.parse_generated(text), tools, prefix))
        print(f"\r{n} of {len(eval_cases)}", end="")
    print("\r" + tally.report(title))
    return tally

before = score("Before training")

In [ ]:
# The teaching itself: the longest step.
from datasets import Dataset
from transformers import DataCollatorForSeq2Seq, Trainer, TrainingArguments
from unsloth import is_bfloat16_supported

FastLanguageModel.for_training(model)
trainer = Trainer(
    model=model,
    train_dataset=Dataset.from_list(lessons),
    data_collator=DataCollatorForSeq2Seq(tokenizer, padding=True, label_pad_token_id=-100),
    args=TrainingArguments(
        output_dir="/tmp/outputs", per_device_train_batch_size=1, gradient_accumulation_steps=8,
        num_train_epochs=EPOCHS, learning_rate=LEARNING_RATE, warmup_ratio=0.05, lr_scheduler_type="cosine",
        weight_decay=0.01, optim="adamw_8bit", fp16=not is_bfloat16_supported(), bf16=is_bfloat16_supported(),
        logging_steps=10, save_strategy="no", report_to="none", seed=3407,
    ),
)
started = time.time()
trainer.train()
print(f"Taught in {(time.time() - started) / 60:.0f} minutes.")

In [ ]:
after = score("After training")

In [ ]:
# One file for llama.cpp (GGUF, Q4_K_M: about 1.1 GB), named for today so LightMoon can tell when it was taught.
name = f"lightmoon-{date.today().isoformat()}"
final = f"{WORK}/{name}.Q4_K_M.gguf"

def sh(command):
    subprocess.run(command, shell=True, check=True)

def new_gguf_files(since):
    found = glob.glob(f"{WORK}/**/*.gguf", recursive=True) + glob.glob("/tmp/**/*.gguf", recursive=True)
    return [f for f in found if os.path.getmtime(f) >= since and "q4_k_m" in os.path.basename(f).lower() and f != final]

started = time.time()
try:
    model.save_pretrained_gguf(f"/tmp/{name}", tokenizer, quantization_method="q4_k_m")
    os.replace(max(new_gguf_files(started), key=os.path.getmtime), final)
except Exception as err:  # Unsloth's own export changes from version to version: llama.cpp's tools, directly
    print(f"Unsloth's export didn't work ({err}); using llama.cpp's own tools instead.")
    model.save_pretrained_merged("/tmp/merged", tokenizer, save_method="merged_16bit")
    if not os.path.isdir("/tmp/llama.cpp"):
        sh("git clone -q --depth 1 https://github.com/ggml-org/llama.cpp /tmp/llama.cpp")
    sh("pip install -q /tmp/llama.cpp/gguf-py sentencepiece protobuf mistral_common")
    sh("python /tmp/llama.cpp/convert_hf_to_gguf.py /tmp/merged --outtype f16 --outfile /tmp/f16.gguf")
    sh("cmake -S /tmp/llama.cpp -B /tmp/llama.cpp/build -DLLAMA_CURL=OFF -DGGML_NATIVE=OFF > /dev/null"
       " && cmake --build /tmp/llama.cpp/build --target llama-quantize -j 4 > /dev/null")
    sh(f"/tmp/llama.cpp/build/bin/llama-quantize /tmp/f16.gguf {final} Q4_K_M")

# Only the model and its report stay in Output.
for leftover in [f"{WORK}/{name}", f"{WORK}/{name}_gguf", f"{WORK}/llama.cpp", f"{WORK}/outputs", f"{WORK}/__pycache__"]:
    shutil.rmtree(leftover, ignore_errors=True)
for leftover in [f"{WORK}/eval.py", *new_gguf_files(started)]:
    if os.path.exists(leftover):
        os.remove(leftover)

with open(f"{WORK}/{name}-report.txt", "w") as report:
    report.write(f"{name}: taught from {len(lessons)} answers in {len(examples)} conversations, {EPOCHS} pass(es), base {BASE_MODEL}.\n\n")
    for tally in (before, after):
        if tally:
            report.write(tally.report("Before training" if tally is before else "After training") + "\n\n")
print(f"Done: {os.path.basename(final)} ({os.path.getsize(final) / 1e9:.2f} GB).")
if IN_COLAB:
    colab_files.download(final)
else:
    print("Download it from Output on the right (the /kaggle/working folder).")

## Then, on your computer

Leave the downloaded `.gguf` file in Downloads and double-click **Start our model** in
`C:\Users\<you>\LightMoon\own-model`: it moves the file into `models` and starts using it. LightMoon's
*Our model on this computer* card then shows the new model and when it was taught, and scores it afresh. Keep
collecting: teaching it again later, with more examples, makes it better.